# A 5|7 grain boundary in Boron Nitride

## 0. Introduction

This notebook builds the 5|7 grain boundary in a Boron Nitride monolayer and relaxes it, following the manuscript:

> **Qiucheng Li, Xiaolong Zou, Mengxi Liu, Jingyu Sun, Yabo Gao, Yue Qi, Xiebo Zhou, Boris I. Yakobson, Yanfeng Zhang, and Zhongfan Liu**
> "Grain Boundary Structures and Electronic Properties of Hexagonal Boron Nitride on Cu(111)"
> *Nano Letters* **2015** 15 (9), 5804-5810
> [DOI: 10.1021/acs.nanolett.5b01852](https://doi.org/10.1021/acs.nanolett.5b01852)

Reproducing the Type III boundary of Figure 3f: a continuous 5-7-5-7 chain of pentagons and heptagons, with B–B and N–N bonds alternating along it. The cell holds two such boundaries, as in the manuscript's calculation. No vacuum is added: the standata monolayer already carries 20 Å, more than the manuscript's 10 Å.

The boundary runs along the lattice vectors (3, 1) of one grain and (1, 3) of the other, written in the 60° basis of the literature: the 32.2° boundary of the manuscript.

## 1. Prepare the Environment
### 1.1. Set up the notebook
Set the following flags to control the notebook behavior.


In [ ]:
# Grain boundary parameters
BOUNDARY_VECTORS = ((3, 1), (1, 3))  # lattice vectors of the two grains along the boundary, 60° basis
CORE = "5|7"  # rings rebuilt along both boundaries
HOMO_BONDS = "alternating"  # B–B and N–N bonds in equal numbers at both boundaries: Type III
NUMBER_OF_PERIODS_ACROSS = 2  # repetitions of each grain across the boundary
VACUUM = 0.0  # in Angstroms

# Relaxation parameters
MACE_MODEL_FAMILY = "MACE-MP-0"
MACE_MODEL = "large"
MACE_DISPERSION = False  # the manuscript uses PBE without a dispersion correction
MACE_DEFAULT_DTYPE = "float64"
MACE_DEVICE = "cpu"
FMAX = 0.01  # eV/Å, the force threshold of the manuscript

# Output parameters
UNRELAXED_MATERIAL_NAME = "h-BN 5-7 grain boundary"
MATERIAL_NAME = "h-BN 5-7 grain boundary, MACE-relaxed"

# Visualization parameters
CELL_REPETITIONS_FOR_VISUALIZATION = [1, 2, 1]

### 1.2. Install packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install`.

In [ ]:
from mat3ra.notebooks_utils.calculators.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.3. Load and preview input material

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials

material = Material.create(Materials.get_by_name_and_categories("Boron_Nitride", "2D"))

## 2. Prepare Material
### 2.1. Select and visualize initial material

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials(material, repetitions=CELL_REPETITIONS_FOR_VISUALIZATION)

## 3. Generate the Grain Boundary
### 3.1. Build the grain boundary

In [ ]:
from mat3ra.made.tools.helpers import create_grain_boundary_linear

grain_boundary = create_grain_boundary_linear(
    material=material,
    boundary_vectors=BOUNDARY_VECTORS,
    core=CORE,
    homo_bonds=HOMO_BONDS,
    number_of_periods_across=NUMBER_OF_PERIODS_ACROSS,
    vacuum=VACUUM,
)

### 3.2. Analyze the grain boundary
The two boundaries lie at x = 0 and at the middle of the cell; for each, the rings other than hexagons along it and its B–B and N–N bonds are printed.

In [ ]:
from collections import Counter

import numpy as np
from mat3ra.made.tools.helpers import RingsMaterialAnalyzer


def print_grain_boundary_checks(grain_boundary):
    rings_analyzer = RingsMaterialAnalyzer(material=grain_boundary)
    boundary_positions = [0.0, grain_boundary.lattice.a / 2]
    _, _, bond_vectors = rings_analyzer.bonds
    print(f"{grain_boundary.basis.number_of_atoms} atoms: {dict(Counter(grain_boundary.basis.elements.values))}")
    for position, arrangement in zip(boundary_positions, rings_analyzer.get_line_arrangements(boundary_positions)):
        print(f"Boundary at x = {position:.2f} Å: rings {arrangement.cores}, homo bonds {arrangement.homo_bonds}")
    print(f"Shortest interatomic distance: {np.linalg.norm(bond_vectors, axis=1).min():.3f} Å")


print_grain_boundary_checks(grain_boundary)

## 4. Relax the Grain Boundary
### 4.1. Build the MACE calculator

In [ ]:
from mat3ra.notebooks_utils.calculators.mlff import create_mlff_calculator

calculator = create_mlff_calculator(
    "mace",
    {
        "family": MACE_MODEL_FAMILY,
        "model": MACE_MODEL,
        "dispersion": MACE_DISPERSION,
        "default_dtype": MACE_DEFAULT_DTYPE,
        "device": MACE_DEVICE,
    },
)

### 4.2. Relax atomic positions at fixed cell

In [ ]:
from mat3ra.made.tools.calculate import calculate_total_energy
from mat3ra.notebooks_utils.workflows.relaxation import relax_material

initial_energy = calculate_total_energy(grain_boundary, calculator)
relaxed_grain_boundary = relax_material(grain_boundary, calculator, fmax=FMAX)
relaxed_energy = calculate_total_energy(relaxed_grain_boundary, calculator)
print(f"Energy change on relaxation: {relaxed_energy - initial_energy:.3f} eV")
print_grain_boundary_checks(relaxed_grain_boundary)

## 5. Preview the grain boundary

In [ ]:
visualize_materials(
    [{"material": grain_boundary, "title": "As built"}, {"material": relaxed_grain_boundary, "title": "MACE-relaxed"}],
    repetitions=CELL_REPETITIONS_FOR_VISUALIZATION,
    viewer="wave",
)

## 6. Pass data to the outside runtime


In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

grain_boundary.name = UNRELAXED_MATERIAL_NAME
relaxed_grain_boundary.name = MATERIAL_NAME
set_materials([grain_boundary, relaxed_grain_boundary])
download_content_to_file(relaxed_grain_boundary.to_json(), f"{relaxed_grain_boundary.name}.json")

## References

[1] Qiucheng Li, Xiaolong Zou, Mengxi Liu, Jingyu Sun, Yabo Gao, Yue Qi, Xiebo Zhou, Boris I. Yakobson, Yanfeng Zhang, Zhongfan Liu, "Grain Boundary Structures and Electronic Properties of Hexagonal Boron Nitride on Cu(111)", Nano Lett. 15(9), 5804-5810 (2015). [DOI: 10.1021/acs.nanolett.5b01852](https://doi.org/10.1021/acs.nanolett.5b01852)

[2] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace